# Dataset 
The dataset `TabularDataset` is built on top of huggingface datasets, which is documented here: https://huggingface.co/docs/datasets/en/index

In contrast to a pure HuggingFace dataset however, `TabularDataset` always returns torch tensors from its `__getitem__` method. These are only converted to numpy when we need to afterwards for working with pure sklearn estimators.

In this notebook we will show how to apply preprocessing, filtering, and on-the-fly transforms to the dataset and show it's indexing and compatability methods for sklearn. 


## Construction

For constructing a dataset we minimally need a path to the directory where data live, the data format they are stored in, and the column in the data that represents the label, or a list thereof. 

In [ ]:
%load_ext autoreload
%autoreload 2

from GalaxySpectrumClassifier import TabularDataset

dataset = TabularDataset(
    path="../data/bronze/default",
    data_format="csv",
    label_columns="source",
)
dataset

We can also construct a dataset from a yaml config file with the same content:

In [ ]:
from pprint import pprint

import yaml

with open("../configs/dataset_example.yaml", "r") as f:
    config = yaml.safe_load(f)

pprint("config file: ")
pprint(config)

dataset = TabularDataset.from_config(config)
dataset

## Indexing 

A single line in the dataset corresponds to a single data point. Indexing hence works as one would expect

In [ ]:
# get a single (the 0th) observation

dataset[0]

notice that we get back a tuple of two torch tensors. The first is the feature tensor, containing the columns of a single observation in the same order they are in the dataset's raw data. The second is the label column, in this case it's called 'source', and it encodes 2 classes: 0 (AGN) and 1 (star formation).

In [ ]:
X, y = dataset[0]

we can slice or use wrap-around indexing too:

In [ ]:
X, y = dataset[10:15]
X, y

In [ ]:
X.shape, y.shape

This gives us rows, i.e., datapoints 10,11,12,13,14. Indexing with -1 gives us the last datapoint, and so on. 

In [ ]:
X, y = dataset[-1]
X, y

We can split the complete dataset into feature and label tensors and  convert them to numpy to use them in a sklearn estimator, which wants numpy arrays:  

In [ ]:
X, y = dataset[:]

X = X.numpy()
y = y.numpy()
X.shape, y.shape, type(X), type(y)

We can also inspect the columns and data sizes of the dataset via the underlying backend object: 

In [ ]:
dataset.backend.column_names

In [ ]:
dataset.backend.num_columns

In [ ]:
dataset.backend.num_rows

Because this is a relatively common thing (sklearn does not train in batches) we have a convenience function for it: 

In [ ]:
from GalaxySpectrumClassifier import TabularDataset, to_xy

dataset = TabularDataset(
    path="../data/bronze/default",
    data_format="csv",
    label_columns="source",
)

X, y = to_xy(dataset)
X.shape, y.shape, type(X), type(y)

## Formatting

Selecting of columns and formatting works via the `set_format` function of HuggingFace Datasets as documented here: https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.set_format  

In [ ]:
from GalaxySpectrumClassifier import TabularDataset

dataset = TabularDataset(
    path="../data/bronze/default",
    data_format="csv",
    label_columns="source",
)

dataset.set_format(
    columns=[
        "12+log(O/H)",
        "OII_3727",
        "source",
    ]
)

X, y = dataset[1]

X, y

The format will include the missing label columns if they are missing, so we always know what the labels are when indexing.

In [ ]:
from GalaxySpectrumClassifier import TabularDataset

dataset = TabularDataset(
    path="../data/bronze/default",
    data_format="csv",
    label_columns="source",
)

dataset.set_format(
    columns=[
        "12+log(O/H)",
        "OII_3727",
        # "source", The label column is missing now!
    ]
)

X, y = dataset[1]  # but the dataset got your back :)
X, y

getting rid of it works as well. 

In [ ]:
dataset.reset_format()
X, y = dataset[1]  # but the dataset got your back :)
X, y

## Filtering and preprocessing

This pre-transform example reads the two committed raw Cloudy grids from the bronze stag and processes them into 'silver' data. The raw `source` field contains strings, so the map function below replaces it with an integer `source_label` (`AGN = 0`, `HII = 1`). Hugging Face materializes the loaded and transformed Arrow data in `data/silver/default` through `cache_dir`.

This is an example of a bronze-to-silver stage data preprocessing step. It delegates parsing, mapping, and caching to `TabularDataset` and Hugging Face Datasets.


In [ ]:
from pathlib import Path

from GalaxySpectrumClassifier import TabularDataset


def encode_source(row):
    """Replace a raw Cloudy source string with a classifier label."""
    labels = {"AGN": 0, "HII": 1}
    try:
        return {"source_label": labels[row["source"]]}
    except KeyError as error:
        raise ValueError(f"Unknown source label: {row['source']!r}") from error


bronze_files = [
    "../data/bronze/default/C17_AGN_alpha08_efrac02_CNfix.dat",
    "../data/bronze/default/C17_POPSTAR_1myr.dat",
]
silver_cache = Path("../data/silver/default")

prepared_dataset = TabularDataset(
    data_format="csv",
    label_columns="source_label",
    pre_transform="__main__.encode_source",
    # Remove the string column so the new integer label gets an integer schema.
    pre_transform_kwargs={"remove_columns": ["source"]},
    hf_dataset_kwargs={
        "data_files": bronze_files,
        "comment": "#",
        "sep": r"\s+",
        "cache_dir": str(silver_cache),
    },
)

print(f"Prepared rows: {len(prepared_dataset)}")
print(f"First encoded label: {prepared_dataset.backend[0]['source_label']}")
print(f"Silver cache: {silver_cache.resolve()}")

The same bronze-to-silver setup can live in YAML. `dataset_preprocessing_example.yaml` names the two tracked bronze files, configures their whitespace/comment parsing, removes the raw string `source` column while mapping it to `source_label`, and writes the Hugging Face cache to `data/silver/default`.

The hook must be importable when the configuration is loaded; here `encode_source` is defined above in `__main__`.


In [ ]:
from pathlib import Path
from pprint import pprint

import yaml

with open("../configs/dataset_preprocessing_example.yaml", "r") as f:
    preprocessing_config = yaml.safe_load(f)

pprint(preprocessing_config)
cache_dir = Path(preprocessing_config["hf_dataset_kwargs"]["cache_dir"]).resolve()
print(f"Silver cache: {cache_dir}")

configured_dataset = TabularDataset.from_config(preprocessing_config)
print(f"Configured rows: {len(configured_dataset)}")
print(f"First encoded label: {configured_dataset.backend[0]['source_label']}")

## Transforming data on the fly before indexing

A regular `transform` is installed with Hugging Face's `with_transform`. It runs whenever rows are retrieved instead of rewriting the prepared Arrow data. The configuration below starts from `preprocessing_config`, so it reloads the same bronze-to-silver pre-transform and reuses its cache in `data/silver/default`; the new transform remains lazy.

`transform_kwargs["columns"]` selects the columns passed to the formatter. `TabularDataset` also includes `source_label` automatically so the target remains available. Hugging Face calls the formatter with a batch dictionary—even when one row is requested—so each value below is a list.


In [ ]:
from GalaxySpectrumClassifier import TabularDataset


def double_selected_oii(batch):
    batch = dict(batch)
    batch["OII_3727"] = [value * 2 for value in batch["OII_3727"]]
    return batch


# no path files here. we are reusing the cache
dataset = TabularDataset(
    data_format="csv",
    hf_dataset_kwargs={
        "cache_dir": "../data/silver/default_fromconfig",
        "comment": "#",
        "data_files": [
            "../data/bronze/default/C17_AGN_alpha08_efrac02_CNfix.dat",
            "../data/bronze/default/C17_POPSTAR_1myr.dat",
        ],
        "sep": "\\s+",
    },
    label_columns="source_label",
    pre_transform="__main__.encode_source",
    pre_transform_kwargs={"remove_columns": ["source"]},
    transform=double_selected_oii,
    transform_kwargs={"columns": ["OII_3727"]},
)

stored_value = prepared_dataset.backend[0]["OII_3727"]
formatted_row = dataset.backend[0]

print(f"Formatted columns: {list(formatted_row)}")
print(f"OII_3727 on retrieval: {stored_value:.4f} -> {formatted_row['OII_3727']:.4f}")
print(f"Stored Arrow value is still: {prepared_dataset.backend[0]['OII_3727']:.4f}")

## Loading the prepared cache directly

The preprocessing step above has already written a mapped Arrow file under `data/silver/default/csv/`. A consumer can load that cached file directly with `data_format="arrow"`; it does not need to rerun the pre-transform.


In [ ]:
silver_cache = Path("../data/silver/default")

cached_dataset = TabularDataset(
    data_format="arrow",
    label_columns="source_label",
    hf_dataset_kwargs={
        "data_files": str(silver_cache / "csv/**/cache-*.arrow"),
    },
)

cached_features, cached_label = cached_dataset[0]
print(f"Cached rows: {len(cached_dataset)}")
print(f"Features: {tuple(cached_features.shape)}, label: {cached_label.item()}")